# 🎓 Fine-Tuning GPT-2 on NYU Q&A Dataset


**Objective:** Fine-tune GPT-2 (a transformer-based language model by OpenAI) on a custom NYU (New York University) dataset so it generates coherent, contextually relevant answers about NYU.

**What you will learn:**
- How to prepare a custom Q&A dataset for language model training
- How to fine-tune GPT-2 using Hugging Face `transformers`
- How to generate text using the fine-tuned model

**References:** [Hugging Face Docs](https://huggingface.co/docs/transformers) | [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

---
> ⚠️ **Before you start:** Go to **Runtime → Change runtime type → T4 GPU** to enable GPU acceleration.

## Step 1 — Verify GPU

In [2]:
!nvidia-smi

Fri Apr 24 06:48:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Install Dependencies

In [3]:
# Installing required libraries
!pip install -q transformers datasets accelerate
print("Libraries installed successfully!")

Libraries installed successfully!


## Step 3 — Upload & Prepare the Training Dataset

We use Q&A pairs from the NYU dataset, wrapped with special tokens `<|startoftext|>` and `<|endoftext|>`.
This teaches GPT-2 the pattern: **Question → Answer**.

> 💡 **Tip for Interns:** The dataset file `nyu_finetune.txt` must be uploaded to your Colab session. You can also expand it with more NYU-related Q&A pairs for better coverage!

In [5]:
import os

DATASET_FILE = "nyu_dataset.txt"   # Output file name after reformatting
SOURCE_FILE  = "nyu_finetune.txt"  # Your uploaded raw Q&A file

# ── Read and reformat into GPT-2 training format ──────────────────────────
# Each Q&A block gets wrapped with <|startoftext|> ... <|endoftext|>
# so the model learns clean start/end boundaries.

with open(SOURCE_FILE, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split on blank lines to get individual Q&A blocks
blocks = [b.strip() for b in raw_text.strip().split("\n\n") if b.strip()]

formatted_lines = []
for block in blocks:
    # Normalise: replace the raw 'Q:' / 'A:' labels with 'Question:' / 'Answer:'
    # so prompts at inference time are consistent.
    block = block.replace("Q:", "Question:", 1).replace("A:", "Answer:", 1)
    formatted_lines.append(f"<|startoftext|>\n{block}\n<|endoftext|>")

with open(DATASET_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(formatted_lines))

print(f"Dataset prepared: {len(formatted_lines)} Q&A pairs → '{DATASET_FILE}'")

Dataset prepared: 49 Q&A pairs → 'nyu_dataset.txt'


## Step 4 — Load GPT-2 Model & Tokenizer

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"   # Options: "gpt2", "gpt2-medium", "EleutherAI/gpt-neo-125M"

print(f"Loading model: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 doesn't have a pad token by default; use EOS token as pad
tokenizer.pad_token    = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# Add special tokens used in the dataset
special_tokens = {"additional_special_tokens": ["<|startoftext|>", "<|endoftext|>"]}
tokenizer.add_special_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))   # Resize embeddings to match vocab

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded! Total parameters: {total_params:.1f}M")

Loading model: gpt2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded! Total parameters: 124.4M


## Step 5 — Tokenize & Prepare Dataset

In [7]:
from datasets import load_dataset

# Load the text dataset
dataset = load_dataset("text", data_files=DATASET_FILE, split="train")
# Filter empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)
print(f"Dataset loaded: {len(dataset)} lines")

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Group tokens into fixed-size blocks for language modeling
BLOCK_SIZE = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_dataset.map(group_texts, batched=True)
print(f"Tokenization complete. Total blocks: {len(lm_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/244 [00:00<?, ? examples/s]

Dataset loaded: 196 lines


Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Tokenization complete. Total blocks: 26


## Step 6 — Configure Training

**Key hyperparameters explained:**
| Parameter | Value | Meaning |
|---|---|---|
| `num_train_epochs` | 100 | How many full passes over the dataset |
| `learning_rate` | 1e-4 | How fast the model updates weights |
| `per_device_train_batch_size` | 2 | Samples processed per GPU step |
| `weight_decay` | 0.01 | Regularization to prevent overfitting |

In [8]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

OUTPUT_DIR = "./nyu_gpt2_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=100,
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,           # Keep only last 2 checkpoints to save disk space
    prediction_loss_only=True,
    fp16=True,                    # Mixed precision — faster on T4/V100 GPU
    report_to="none",             # Disable W&B logging
)

# Data collator handles dynamic padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False    # GPT-2 is causal LM, not masked LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=data_collator,
)

print("Trainer configured and ready!")

Trainer configured and ready!


## Step 7 — Train the Model

> ⏳ Training takes approximately **3–6 minutes** on a T4 GPU with this dataset size.  
> Watch the **loss** value — it should decrease over time. Lower loss = better model.

In [9]:
print("Starting training...\n")
trainer.train()
trainer.state.log_history[-1]
print("\nTraining complete!")

Starting training...



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.493399
20,3.021064
30,2.588544
40,2.137190
50,1.803595
60,1.485234
70,1.157230
80,0.978866
90,0.725170
100,0.561642


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!


## Step 8 — Save the Fine-Tuned Model

In [10]:
SAVE_DIR = "./nyu_gpt2_finetuned"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to '{SAVE_DIR}'")
!ls -lh {SAVE_DIR}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to './nyu_gpt2_finetuned'
total 479M
-rw-r--r-- 1 root root  962 Apr 24 07:04 config.json
-rw-r--r-- 1 root root  118 Apr 24 07:04 generation_config.json
-rw-r--r-- 1 root root 475M Apr 24 07:04 model.safetensors
-rw-r--r-- 1 root root  373 Apr 24 07:04 tokenizer_config.json
-rw-r--r-- 1 root root 3.4M Apr 24 07:04 tokenizer.json


## Step 9 — Generate Text from Fine-Tuned Model

Try different prompts and observe how the model generates relevant text about NYU!

In [11]:
from transformers import pipeline

# Load the saved model into a text-generation pipeline
generator = pipeline(
    "text-generation",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0   # 0 = GPU, -1 = CPU
)

print("Generator pipeline ready!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generator pipeline ready!


In [14]:

prompts_to_test = [
    "Question: What is New York University?\nAnswer:",
    "Question: When was New York University founded?\nAnswer:",
    "Question: What is the Tisch School of the Arts known for?\nAnswer:",
    "Question: How many Nobel Laureates are associated with NYU?\nAnswer:",
    "Question: What is the acceptance rate at NYU?\nAnswer:",
]

for prompt in prompts_to_test:
    print("=" * 60)
    print(f"PROMPT: {prompt.strip()}")
    outputs = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.6,        # Lower = more focused output
        top_k=40,               # Limits to top 40 token choices
        top_p=0.9,              # Nucleus sampling
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0]["generated_text"].replace(prompt, "").strip()
    print(f"GENERATED: {generated}")
    print()

Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Question: What is New York University?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: NYU (New York University) are a private research university located in New York City. Founded in 1831, they are one of the largest private universities in the United States by enrollment. NYU have been involved with various social causes and organizations, including the Tisch School for Creative Studies, the Stern School Of Business, the Violets Institute, the Thomas Jefferson College of Nursing—in which Richard

PROMPT: Question: When was New York University founded?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: NewYork University was founded on April 18, 1831 by a group of prominent merchants including traders from the United States, and bankers. A key visionary among the founders was Albert Gallatin, former US Secretary at Treasury under President Thomas Jefferson. They believed it needed to respond quickly to rising crime in cities like Manhattan and Abu Dhabi, as wellQuestion 5 Did NYU ever have a campus in the

PROMPT: Question: What is the Tisch School of the Arts known for?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: The Tiffin School and the university's oldest and most respected undergraduate programs are called The Courant Institute. The Tishman School, founded in 1835 by renowned architect Stanford White, is a major private school in New York City with students enrolled across 20 schools and colleges. The Violets are used to organize dance classes and other educational opportunities. There have been multiple Nobel Laureates from the United

PROMPT: Question: How many Nobel Laureates are associated with NYU?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: Past and present faculty and alumni of NYU include 39 Nobel Prize winners. Three have served as heads of state, including Lady Gaga (singer), Adam Sandler (actor and comedian), Alec Baldwin ("comedian"), Idina Menzelbaum "actress and singer," Alan Greenspan Jr., Childish Gambino (musician and filmmaker), Aziz Ansari ("director), the Olsen twins and

PROMPT: Question: What is the acceptance rate at NYU?
Answer:
GENERATED: The acceptance rates at New York University are approximately 7 to 1. This translates to an average of around 85%. There are also students enrolled at NYU Langone Medical Center, which is located in Abu Dhabi, United Arab Emirates. The total faculty stands at over 5,700 according and is one of the largest employers in New Jersey City, with more than 19,000 employees. The campus lacks traditional boundaries



## Step 10 — Experiment & Reflect

Use the cell below to freely test your model with any prompt.

In [13]:
my_prompt = "Question: What is New York University?\nAnswer:"

outputs = generator(
    my_prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.3,
    num_return_sequences=3,
    pad_token_id=tokenizer.eos_token_id,
)

print(f"Prompt: {my_prompt}\n")
for i, out in enumerate(outputs, 1):
    generated = out["generated_text"].replace(my_prompt, "").strip()
    print(f"[Output {i}]: {generated}\n")

Passing `generation_config` together with generation-related arguments=({'temperature', 'top_k', 'top_p', 'repetition_penalty', 'max_new_tokens', 'num_return_sequences', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Question: What is New York University?
Answer:

[Output 1]: NYU (NYU) are a private research university located in New York City. Founded in 1831, they are one of the largest private universities in the United States by enrollment. NYU have campuses in New York, Abu Dhabi and Shanghai as wellas academic centers in over 25 countries. The university's athletic teams are called the Violets. The mascot is the Bobcat. The admissions policy at NYUniversity differs from that used before. Students must first be admitted to NYU campus. They do

[Output 2]: NYU (NYU) refers to itself as the first comprehensive liberal arts and research campus to be built and operated outside the United States by an American university. Located in Abu Dhabi, United Arab Emirates it offers undergraduate and graduate degrees and is a full degree-granting portal campus of NYU. The university also has a degree-optional admissions policy, allowing students to study across NYU's three campuses in New York City,

---
## Summary

| Step | What happened |
|------|---------------|
| Dataset | Reformatted NYU Q&A pairs with GPT-2 special tokens |
| Model | Loaded pre-trained GPT-2 (124M parameters) from Hugging Face |
| Tokenization | Converted text to token IDs in fixed-size blocks |
| Training | Fine-tuned GPT-2 on NYU data for 100 epochs |
| Inference | Generated contextual answers about NYU from a given prompt |

## 💡 Ideas to Extend This Project
- Add **more Q&A pairs** on NYU departments, research, or campus life for richer generation
- Try **`gpt2-medium`** (345M params) for better quality (needs more VRAM)
- Try **`EleutherAI/gpt-neo-125M`** as an open-source alternative
- Build a simple **Gradio UI** to let users ask questions about NYU
- Evaluate output quality using **BLEU score** or **perplexity**

---
